# Non-Derivable Itemsets
Your goal here is to write a program to compute whether an itemset is derivable or not. The program should take as input the following two files:

FILE1: A list of itemsets with their support values (one per line). See the file: itemsets.txt (the format is "itemset - support"; one per line)

FILE2: A list of itemsets (one per line), whose support bounds have to be derived. See the file: ndi.txt

Your program should output for each itemset in FILE2 the following info:

itemset: [l,u] derivable/non-derivable

where l and u are the lower and upper-bounds on the support.

In [1]:
import numpy as np
from collections import defaultdict
from itertools import combinations

In [2]:
def nditem(f1='/content/itemsets.txt', f2='/content/ndi.txt'):
  """
  Compute whether an itemset is derivable or not given data from file1 and file2
  * Args:
      - file1: A list of itemsets with their support values of the format "itemset - support"; one per line
      - file2: A list of itemsets (one per line), whose support bounds have to be derived
  * Return:
      itemset: [l,u] derivable/non-derivable for each itemset in file2 where l and u are the lower and upper-bounds on the support.
  """

  # read f1
  with open(f1, 'r') as f:
    lines = f.read().splitlines()
    tid_num = int(lines[0].split('-')[-1])
    lines = lines[1:]

  supports = dict()
  supports[()] = tid_num

  for line in lines:
    itemset, support = line.split('-')
    support = int(support.strip())
    itemset = tuple(np.array(itemset.split()).astype(int))
    supports[itemset] = int(support)

  #  read f2
  with open(f2, 'r') as f:
    lines = f.read().splitlines()

  itemsets = [np.array(line.split()).astype(int) for line in lines]

  def cal_IE(Y, X):
    """
    Calculate IE(Y) given an itemset X and Y is a subset of X
    """
    X = np.array(X)
    Y = np.array(Y)

    Z = np.setdiff1d(X, Y)
    zlen = Z.shape[0]
    ie = 0

    for i in range(zlen):
      for subset in combinations(Z, i):
        W = tuple(np.union1d(Y, np.array(subset)))

        if (zlen - i + 1) % 2 == 0:
          ie += supports[W]

        else:
          ie -= supports[W]

    return ie

  def support_bounds(itemset):
    """
    Calculate lower and upper bound for support
    """

    ub = np.inf
    lb = -1

    n = len(itemset)

    for i in range(n+1):
      for subset in combinations(itemset, i):
        ie = cal_IE(subset, itemset)

        if (n-i) % 2 == 1:
          ub = min(ub, ie)

        else:
          lb = max(lb, ie)

    return lb, ub

  for itemset in itemsets:
    lb, ub = support_bounds(itemset)
    print(f"{' '.join(itemset.astype(str))}: [{lb}, {ub}] {'derivable' if lb == ub else 'non-derivable'}")

In [3]:
nditem()

29 34 40 52 62: [2888, 2888] derivable
7 29: [3061, 3076] non-derivable
29 48 58: [2997, 2997] derivable
7 29 36 40 52 58 60: [2890, 2890] derivable
5 40 52 60: [2893, 2893] derivable
7 36 40 58: [2952, 2952] derivable
36 40 52 58 60 66: [2888, 2888] derivable
